# Dallas TL Attribution Run

This notebook previews the next-token prediction for:

```text
Fact: the capital of the state containing Dallas is
```

Then it runs the TransformerLens-backed attribution path through `BiologyAttributionRunner`, which imports `biology_server_t_lens` for the replacement-model forward and attribution pass. It saves the resulting attribution graph JSON plus an optional compact `.pt` summary.

In [1]:
from google.colab import drive, runtime

drive.mount("/content/drive")

## 1. Repo And Dependency Setup

Run this in a GPU Colab runtime. A high-memory A100 is the intended target for the default `BATCH_SIZE=128` / `MAX_FEATURE_NODES=300` settings.

In [2]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

# 1. Define repository remote URL, target branch, and the local clone destination
REPO_URL = "https://github.com/rowan-dauria/llm-biology.git"
BRANCH = "bio-serv-transformer-lens"
DEFAULT_COLAB_REPO = Path("/content/llm-biology")

# 2. Always clean up any existing repository folder to guarantee a fresh download
if DEFAULT_COLAB_REPO.exists():
    print(f"[SETUP] Removing existing folder {DEFAULT_COLAB_REPO} to ensure a fresh clone...")
    try:
        shutil.rmtree(DEFAULT_COLAB_REPO)
    except Exception as e:
        raise RuntimeError(
            f"[SETUP ERROR] Failed to clean up existing folder '{DEFAULT_COLAB_REPO}': {e}. "
            "Please manually delete the folder or restart your Jupyter kernel."
        ) from e

# 3. Run git clone and log progress loudly; fail loudly with instructions if it fails
print(f"[SETUP] Cloning branch '{BRANCH}' from '{REPO_URL}' to '{DEFAULT_COLAB_REPO}'...")
try:
    subprocess.check_call(["git", "clone", "--branch", BRANCH, REPO_URL, str(DEFAULT_COLAB_REPO)])
    print("[SETUP] Git clone completed successfully!")
except Exception as e:
    raise RuntimeError(
        f"[SETUP ERROR] Git clone failed for '{REPO_URL}' on branch '{BRANCH}'.\n"
        "Verify that your network connection is active and that the branch has been pushed to GitHub."
    ) from e

# 4. Point working directory and python sys.path search to the freshly cloned repo
repo_dir = DEFAULT_COLAB_REPO
os.chdir(repo_dir)
if str(repo_dir) not in sys.path:
    sys.path.insert(0, str(repo_dir))

# 5. Add local circuit-tracer reference to search path if present locally in parent directory
local_circuit_tracer = repo_dir.parent / "circuit-tracer"
if local_circuit_tracer.exists() and str(local_circuit_tracer) not in sys.path:
    sys.path.insert(0, str(local_circuit_tracer))

print(f"[SETUP] Using active repo directory: {repo_dir}")
print(f"[SETUP] Python executable: {sys.executable}")

In [3]:
IN_COLAB = "COLAB_RELEASE_TAG" in os.environ
INSTALL_DEPS = IN_COLAB

if INSTALL_DEPS:
    packages = [
        "accelerate",
        "circuit-tracer",
        "einops",
        "fsspec",
        "huggingface-hub<1.0",
        "safetensors",
        "sentencepiece",
        "transformer-lens>=2.16.0",
        "transformers>=4.56.0,<=4.57.3",
        "tqdm",
        "zstandard",
    ]
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *packages])
else:
    print("Skipping dependency install outside Colab. Set INSTALL_DEPS=True to force it.")

In [4]:
# Keep this notebook compatible with both circuit-tracer factory names.
import importlib
import sys

slt_module = importlib.import_module("circuit_tracer.transcoder.single_layer_transcoder")
if not hasattr(slt_module, "load_transcoder"):
    if not hasattr(slt_module, "load_relu_transcoder"):
        raise ImportError(
            "circuit_tracer.transcoder.single_layer_transcoder has neither "
            "load_transcoder nor load_relu_transcoder"
        )
    slt_module.load_transcoder = slt_module.load_relu_transcoder
    print("Added load_transcoder alias for this notebook runtime.")
else:
    print("circuit-tracer already exposes load_transcoder.")

# Drop any half-imported modules left by a failed earlier run in this kernel.
for module_name in list(sys.modules):
    if module_name == "biology_server" or module_name.startswith("biology_server."):
        del sys.modules[module_name]

In [5]:
# Optional: use a Colab secret named HF_TOKEN if you need authenticated HF downloads.
try:
    from google.colab import userdata
    from huggingface_hub import login

    hf_token = userdata.get("HF_TOKEN")
    if hf_token:
        login(token=hf_token)
        print("Logged in to Hugging Face with HF_TOKEN.")
    else:
        print("No HF_TOKEN Colab secret found; continuing unauthenticated.")
except Exception as exc:
    print(f"HF login skipped: {exc}")

In [6]:
import torch

print(f"torch={torch.__version__}")
print(f"cuda_available={torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"gpu={torch.cuda.get_device_name(0)}")
    props = torch.cuda.get_device_properties(0)
    print(f"gpu_memory_gb={props.total_memory / 1024**3:.1f}")

## 2. Configure The Run

`USE_CHAT_TEMPLATE=False` is intentional for this factual completion prompt. The preview target token is passed into graph generation so the attribution run targets exactly the token shown in the preview.

In [7]:
import inspect
from datetime import datetime
from pathlib import Path

import biology_server.attribution as attribution_module
import biology_server_t_lens
from biology_server.attribution import DEFAULT_LAYERS, BiologyAttributionRunner

assert "biology_server_t_lens" in inspect.getsource(BiologyAttributionRunner._ensure_loaded)
assert "biology_server_t_lens" in inspect.getsource(BiologyAttributionRunner.generate_graph)
print(f"Using TL backend package: {biology_server_t_lens.__file__}")

PROMPT = "Fact: the capital of the state containing Dallas is"
SLUG = f"{datetime.now().strftime('%Y-%m-%d-%H-%M-%S')}-dallas-state-capital-tl"
USE_CHAT_TEMPLATE = False

LAYERS = DEFAULT_LAYERS
MAX_FEATURE_NODES = 3000
BATCH_SIZE = 32

OUTPUT_DIR = Path("data/colab_attribution_graphs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SAVE_PT_PATH = None

DRIVE_TOPK_DIR = "/content/drive/MyDrive/mphil-project/llm-biology/data/feature_topk/150k-pile"
# ! This is preventing the creation of new feature sidecars
# ! Hacky monkey patch
# Feature-example sidecars need data/feature_topk/150k-pile. They are not needed
# for the raw graph artifact, so skip them by default in Colab.
SKIP_FEATURE_EXAMPLES = False  # uploaded the feature topk's to drive
if SKIP_FEATURE_EXAMPLES:

    def _skip_feature_examples(**_kwargs):
        print("[INFO] skipping feature-example sidecars")
        return {}

    attribution_module.build_feature_examples = _skip_feature_examples

runner = BiologyAttributionRunner(
    layers=LAYERS,
    graph_file_dir=OUTPUT_DIR,
    batch_size=BATCH_SIZE,
    max_feature_nodes=MAX_FEATURE_NODES,
    topk_dir=DRIVE_TOPK_DIR,
)

print(f"prompt={PROMPT!r}")
print(f"slug={SLUG}")
print(f"layers={LAYERS}")
print(f"output_dir={OUTPUT_DIR.resolve()}")

## 3. Preview The Logit Prediction

In [8]:
preview = runner.preview(
    PROMPT,
    slug=SLUG,
    use_chat_template=USE_CHAT_TEMPLATE,
)

print("Prompt tokens:")
for idx, token in enumerate(preview.prompt_tokens):
    print(f"  {idx:02d}: {token!r}")

print("\nTarget token:")
print(
    f"  id={preview.target_token_id} "
    f"text={preview.target_token_str!r} "
    f"prob={preview.target_token_prob:.6f}"
)

print("\nTop preview tokens:")
for item in preview.top_tokens:
    print(f"  id={item.token_id:<8} p={item.prob:.6f} text={item.token!r}")

## 4. Run TransformerLens Attribution And Save The Graph

In [9]:
result = runner.generate_graph(
    PROMPT,
    slug=SLUG,
    target_token_id=preview.target_token_id,
    preview_top_token_id=preview.target_token_id,
    preview_top_token=preview.target_token_str,
    preview_top_token_prob=preview.target_token_prob,
    use_chat_template=USE_CHAT_TEMPLATE,
    save_pt=SAVE_PT_PATH,
)

print("Saved attribution graph:")
print(f"  graph_json={result.graph_path}")
print(f"  compact_pt={result.pt_path}")
print(f"  target={result.target_token_str!r} p={result.target_token_prob:.6f}")
print(f"  feature_nodes={len(result.selected_features)}")
print(f"  links={len(result.links)}")
print(f"  logit_targets={len(result.logit_targets)}")

## 5. Inspect And Download Saved Artifacts

In [10]:
import json

graph_path = Path(result.graph_path)
with graph_path.open(encoding="utf-8") as handle:
    graph = json.load(handle)

print(json.dumps(graph["metadata"], indent=2))
print(f"nodes={len(graph['nodes'])}")
print(f"links={len(graph['links'])}")
print(f"json_size_mb={graph_path.stat().st_size / 1024**2:.2f}")
if result.pt_path is not None:
    pt_path = Path(result.pt_path)
    print(f"pt_size_mb={pt_path.stat().st_size / 1024**2:.2f}")

In [11]:
import json
import shutil
from pathlib import Path

src_dir = Path("/content/llm-biology/data/colab_attribution_graphs")
dst_dir = Path("/content/drive/MyDrive/mphil-project/colab_attribution_graphs")
dst_dir.mkdir(parents=True, exist_ok=True)

# Copy graph JSON.
graph_src = src_dir / f"{SLUG}.json"
graph_dst = dst_dir / f"{SLUG}.json"
shutil.copy2(graph_src, graph_dst)
print("copied graph:", graph_dst, graph_dst.stat().st_size)

# Copy optional .pt.
if SAVE_PT_PATH is not None:
    pt_src = src_dir / f"{SLUG}.pt"
    pt_dst = dst_dir / f"{SLUG}.pt"
    shutil.copy2(pt_src, pt_dst)
    print("copied pt:", pt_dst, pt_dst.stat().st_size)

In [12]:
# Copy frontend feature/window sidecars.
scan_name = "qwen3-4b-transcoders"
features_src = src_dir / "features" / scan_name
features_dst = dst_dir / "features" / scan_name

if not features_src.exists():
    raise FileNotFoundError(
        f"Missing feature sidecars: {features_src}. "
        "Check that SKIP_FEATURE_EXAMPLES=False and topk_dir is correct."
    )

shutil.copytree(features_src, features_dst, dirs_exist_ok=True)
sidecars = list(features_dst.glob("*.json"))
print(f"copied feature sidecars: {len(sidecars)} files -> {features_dst}")

# Verify every graph feature node has a sidecar.
graph = json.loads(graph_src.read_text())
missing = []
for node in graph["nodes"]:
    if node.get("feature_type") == "cross layer transcoder":
        paired = int(node["feature"])
        if not (features_dst / f"{paired}.json").exists():
            missing.append((node["node_id"], paired))

if missing:
    raise RuntimeError(f"Missing {len(missing)} feature sidecars, first few: {missing[:10]}")

print("all graph feature nodes have sidecars")

In [13]:
runtime.unassign()